# Pilot run — Qwen2-Audio-7B-Instruct

Runs the full pilot after the smoke test (`smoke_test.ipynb`) has passed:

1. **Main pilot**: 15 fictitious facts + 8 knowledge-neutral control facts, 9 conditions (incl. a question-only baseline).
2. **Complexity dataset**: 12 fictitious facts at 4 sentence-difficulty levels.

Every item is scored with two readouts: **Yes/No** (primary: one question per answer value, so no option order)
and **A/B** (comparable to Paper 2; conflict items in both orders, split into content and position effects).

**Before running:** start Jupyter from the repo in an environment with a CUDA GPU (see the smoke test).
Total: 781 stimuli; each gets one A/B forward pass, one short greedy generation and (except the second A/B order) two Yes/No forward passes.
Runs are resumable: if the kernel dies, re-run the same cell and finished stimuli are skipped.

## 1. Project root and GPU check

In [ ]:
import os, sys, torch
from pathlib import Path

# the kernel may start in notebooks/ -> move to the repo root so all relative paths work
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/eval/run_model.py").exists())
os.chdir(REPO)
print("working in", REPO)

assert torch.cuda.is_available(), "No GPU visible to this kernel (check CUDA_VISIBLE_DEVICES / the driver)"
gpus = [torch.cuda.get_device_properties(i) for i in range(torch.cuda.device_count())]
VRAM_GB = sum(g.total_memory for g in gpus) / 1e9
for g in gpus:
    print(f"{g.name}: {g.total_memory / 1e9:.1f} GB")
# 7B in fp16 needs ~16 GB of weights plus activations; device_map="auto" splits it across GPUs
QUANTIZATION = "none" if VRAM_GB >= 20 else "4bit"
# native bf16 needs compute capability >= 8.0 (Ampere or newer); older GPUs only emulate it -> float16
DTYPE = "bfloat16" if min(torch.cuda.get_device_capability(i)[0] for i in range(len(gpus))) >= 8 else "float16"
print(f"total {VRAM_GB:.1f} GB -> quantization: {QUANTIZATION} | dtype: {DTYPE}")

## 2. Install dependencies
Installs `requirements.txt` into the environment of **this kernel** (`sys.executable`). Skip it if the environment is already set up.

In [ ]:
INSTALL = False
if INSTALL:
    !{sys.executable} -m pip install -q -r requirements.txt
import transformers; print("transformers", transformers.__version__)

## 3. Local config
Copies `configs/pilot.yaml` to `configs/local.yaml` (git-ignored) with the quantization / dtype chosen for this GPU.

In [ ]:
import yaml
cfg = yaml.safe_load(open("configs/pilot.yaml"))
cfg["model"]["quantization"] = QUANTIZATION
cfg["model"]["dtype"] = DTYPE
yaml.safe_dump(cfg, open("configs/local.yaml", "w"), sort_keys=False)
print(yaml.safe_dump(cfg["model"]), "eval:", cfg["eval"]["uq"], "| greedy_check:", cfg["eval"]["greedy_check"])

## 4. Main pilot: 15 fictitious + 8 control facts

253 stimuli (conflict items in both A/B orders, plus a question-only baseline), each scored with two readouts:
**Yes/No** (primary, no option order) and **A/B** (comparable to Paper 2). Resumable: re-run the cell after a crash.

In [ ]:
!{sys.executable} -m src.eval.run_model --config configs/local.yaml --out results/pilot_run_1

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("results/pilot_run_1/summary.md").read()))

In [ ]:
for kind in ["fictitious", "control"]:
    for readout in ["yes_no", "ab"]:
        print(f"===== {kind} | {readout}")
        !{sys.executable} -m src.analysis.h1_h2_h3 --results results/pilot_run_1 --kind {kind} --readout {readout}
display(Image("results/pilot_run_1/audio_conf_vs_conflict_fictitious_yes_no.png"))

## 5. Complexity dataset: 12 facts x 4 levels (L0 baseline, L1 extra info, L2 embedding, L3 lure)

528 stimuli. The key question: does the conflict text-follow rate rise as audio-only confidence falls?

In [ ]:
!{sys.executable} -m src.eval.run_model --config configs/local.yaml --stimuli data/complexity/stimuli.jsonl --out results/complexity_run_1

In [ ]:
!{sys.executable} -m src.analysis.complexity_sweep --results results/complexity_run_1 --readout yes_no
display(Image("results/complexity_run_1/complexity_sweep_yes_no.png"))
!{sys.executable} -m src.analysis.complexity_sweep --results results/complexity_run_1 --readout ab

In [ ]:
display(Markdown(open("results/complexity_run_1/summary.md").read()))

## 6. Results
Everything is written to `results/pilot_run_1/` and `results/complexity_run_1/` in the repo (`results/` is git-ignored).